In [1]:
import arcpy
import os
import math

In [2]:
gdb = r"D:\Documents\CityEngine\Default Workspace\Area_estudio\models\estudio.gdb"
multipatch_edif = "Area_estudio_ProcedurallyGeneratedMultipatches"      
multipatch_lum = "Luminarias_ProcedurallyGeneratedMultipatches"      
multipatch_sem = "Red_Semaforica2_ProcedurallyGeneratedMultipatches"  
multipatch_troncales = "troncales_ProcedurallyGeneratedMultipatches"    
puntos_lum = "luminarias_pt"               
puntos_sem = "semaforos_pt"              

In [3]:
salida_candidatos = "candidatos_cam_temp"    
sr = arcpy.SpatialReference(32618) 

In [4]:
dist_sample_edges = 5.0

In [5]:
arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

In [6]:
if arcpy.Exists(salida_candidatos):
    arcpy.Delete_management(salida_candidatos)

In [7]:
arcpy.CreateFeatureclass_management(
    out_path=gdb,
    out_name=salida_candidatos,
    geometry_type="POINT",
    spatial_reference=sr,
    has_m="DISABLED",
    has_z="ENABLED"
)

<Result 'D:\\Documents\\CityEngine\\Default Workspace\\Area_estudio\\models\\estudio.gdb\\candidatos_cam_temp'>

In [8]:
arcpy.AddField_management(salida_candidatos, "tipo_struct", "TEXT", field_length=50) 
arcpy.AddField_management(salida_candidatos, "source_id", "LONG")                    

<Result 'D:\\Documents\\CityEngine\\Default Workspace\\Area_estudio\\models\\estudio.gdb\\candidatos_cam_temp'>

In [9]:
# Helper: crear singlepart de multipatch (si aplica)
def ensure_singlepart(input_fc, out_name):
    """Convierte a singlepart si es multipart. Devuelve path a singlepart fc."""
    out_path = os.path.join(gdb, out_name)
    if arcpy.Exists(out_path):
        arcpy.Delete_management(out_path)
    arcpy.MultipartToSinglepart_management(input_fc, out_path)
    return out_path


In [10]:
if not arcpy.Exists("edificios_singlepart"):
    edif_sp = ensure_singlepart(multipatch_edif, "edificios_singlepart")
    
if not arcpy.Exists("troncales_singlepart2"):
    edif_sp = ensure_singlepart(multipatch_troncales, "troncales_singlepart2")

In [11]:
offset = 0.5
directions = [
    (offset, offset),   
    (offset, -offset),  
    (-offset, offset),   
    (-offset, -offset)
]

directions_center = [
    (offset, 0),   
    (0, offset),  
    (-offset, 0),   
    (0, -offset)
]

insert = arcpy.da.InsertCursor(salida_candidatos, ["SHAPE@", "tipo_struct", "source_id"])
with arcpy.da.SearchCursor("edificios_singlepart", ["OBJECTID", "SHAPE@"]) as scur:
    for oid, geom in scur:
        # Recopilar todos los vértices con sus Z
        max_z = -999999
        verts = []  # list of (x,y,z)
        # iterar partes y puntos
        for part in geom:
            for pnt in part:
                if pnt is None:
                    continue
                # Manejar geometría 3D: pnt.Z puede no existir? pnt tiene Z normalmente en multipatch
                z = pnt.Z if hasattr(pnt, "Z") else None
                if z is None:
                    # si no hay Z en el vértice, intentamos obtener Z desde array de vertices como 0
                    z = 0.0
                verts.append((pnt.X, pnt.Y, z))
                if z > max_z:
                    max_z = z

        if len(verts) == 0:
            continue

        # seleccionar vértices próximos a max_z (tolerancia pequeña para planos inclinados)
        tol_z = 0.2  # metros de tolerancia para considerar vértices de azotea (ajusta si tus datos son ruidosos)
        roof_verts = [v for v in verts if abs(v[2] - max_z) <= tol_z]

        # Si no hay vértices exactamente en max_z (caso raro), tomar los que tengan Z>percentil 95
        if len(roof_verts) == 0:
            zs = sorted([v[2] for v in verts])
            if len(zs) >= 4:
                threshold = zs[int(len(zs)*0.95)]
            else:
                threshold = zs[-1]
            roof_verts = [v for v in verts if v[2] >= threshold - tol_z]

        # 1a) Punto central de azotea (centroid de los vértices de techo)
        # avg_x = sum(v[0] for v in roof_verts) / len(roof_verts)
        # avg_y = sum(v[1] for v in roof_verts) / len(roof_verts)
        # avg_z = sum(v[2] for v in roof_verts) / len(roof_verts)
        
        
        # for dx, dy in directions_center:
        #         x = avg_x + dx
        #         y = avg_y + dy
        #         pnt_geom = arcpy.PointGeometry(arcpy.Point(x, y, avg_z), sr, True)
        
        #         insert.insertRow([pnt_geom, "edificio", oid])

        def distance(a, b):
            return math.hypot(a[0]-b[0], a[1]-b[1])

        kept = []
        for v in roof_verts:
            if all(distance(v, k) >= dist_sample_edges for k in kept):
                kept.append(v)

        for v in kept:
            for dx, dy in directions:
                x = v[0] + dx
                y = v[1] + dy
                p = arcpy.PointGeometry(arcpy.Point(x, y, v[2]), sr, True)
                insert.insertRow([p, "edificio", oid])
del insert
arcpy.AddMessage("Puntos candidatos generados para edificios: {}".format(salida_candidatos))

In [12]:
arcpy.management.MakeFeatureLayer(salida_candidatos, "puntos_layer")

<Result 'puntos_layer'>

In [13]:
arcpy.management.SelectLayerByLocation(
    "puntos_layer",
    "INTERSECT_3D",
    "edificios_singlepart"
)

<Result 'puntos_layer'>

In [14]:
selected_count = int(arcpy.management.GetCount("puntos_layer")[0])
print(f"Número de elementos seleccionados: {selected_count}")

Número de elementos seleccionados: 51839


In [15]:
arcpy.management.DeleteFeatures("puntos_layer")

<Result 'puntos_layer'>

In [16]:
offset = 0.5
directions = [
    (offset, offset),   
    (offset, -offset),  
    (-offset, offset),   
    (-offset, -offset)
]

directions_center = [
    (offset, 0),   
    (0, offset),  
    (-offset, 0),   
    (0, -offset)
]

insert = arcpy.da.InsertCursor(salida_candidatos, ["SHAPE@", "tipo_struct", "source_id"])
with arcpy.da.SearchCursor("troncales_singlepart", ["OBJECTID", "SHAPE@"]) as scur:
    for oid, geom in scur:
        # Recopilar todos los vértices con sus Z
        max_z = -999999
        verts = []  # list of (x,y,z)
        # iterar partes y puntos
        for part in geom:
            for pnt in part:
                if pnt is None:
                    continue
                # Manejar geometría 3D: pnt.Z puede no existir? pnt tiene Z normalmente en multipatch
                z = pnt.Z if hasattr(pnt, "Z") else None
                if z is None:
                    # si no hay Z en el vértice, intentamos obtener Z desde array de vertices como 0
                    z = 0.0
                verts.append((pnt.X, pnt.Y, z))
                if z > max_z:
                    max_z = z

        if len(verts) == 0:
            continue

        roof_verts = [v for v in verts]

        # Si no hay vértices exactamente en max_z (caso raro), tomar los que tengan Z>percentil 95
        if len(roof_verts) == 0:
            zs = sorted([v[2] for v in verts])
            if len(zs) >= 4:
                threshold = zs[int(len(zs)*0.95)]
            else:
                threshold = zs[-1]
            roof_verts = [v for v in verts if v[2] >= threshold - tol_z]

        # 1a) Punto central de azotea (centroid de los vértices de techo)
        avg_x = sum(v[0] for v in roof_verts) / len(roof_verts)
        avg_y = sum(v[1] for v in roof_verts) / len(roof_verts)
        avg_z = sum(v[2] for v in roof_verts) / len(roof_verts)
        
        # for dx, dy in directions_center:
        #         x = avg_x + dx
        #         y = avg_y + dy
        #         pnt_geom = arcpy.PointGeometry(arcpy.Point(x, y, avg_z + offset), sr, True)
        pnt_geom = arcpy.PointGeometry(arcpy.Point(avg_x, avg_y, avg_z + offset), sr, True)
        
        insert.insertRow([pnt_geom, "troncal", oid])

        # 1b) Puntos a lo largo de la azotea: añadir vértices de techo espaciados por dist_sample_edges
        # Para simplicidad, añadimos los vértices de roof_verts, evitando duplicados cercanos
        def distance(a, b):
            return math.hypot(a[0]-b[0], a[1]-b[1])

        kept = []
        for v in roof_verts:
            #if all(distance(v, k) >= dist_sample_edges for k in kept):
            kept.append(v)

        for v in kept:
            # for dx, dy in directions:
            #     x = v[0] + dx
            #     y = v[1] + dy
            #     p = arcpy.PointGeometry(arcpy.Point(x, y, v[2] + offset), sr, True)
            p = arcpy.PointGeometry(arcpy.Point(v[0], v[1], v[2] + offset), sr, True)
            insert.insertRow([p, "troncal", oid])
del insert
arcpy.AddMessage("Puntos candidatos generados para troncales: {}".format(salida_candidatos))

In [17]:
arcpy.management.SelectLayerByLocation(
    "puntos_layer",
    "INTERSECT_3D",
    "troncales_singlepart"
)

<Result 'puntos_layer'>

In [18]:
selected_count = int(arcpy.management.GetCount("puntos_layer")[0])
print(f"Número de elementos seleccionados: {selected_count}")

Número de elementos seleccionados: 0


In [19]:
arcpy.management.DeleteFeatures("puntos_layer")

<Result 'puntos_layer'>

In [20]:
arcpy.management.Delete("puntos_layer")

<Result 'true'>

In [21]:
def add_point_structures(input_fc, tipo_label, id_field="OBJECTID"):
    """Añade a la salida los puntos de estructuras; si son multipatch, toma su centro y Z mínimo/medio."""
    if not arcpy.Exists(input_fc):
        arcpy.AddWarning("No existe: {}".format(input_fc))
        return
    insert = arcpy.da.InsertCursor(salida_candidatos, ["SHAPE@", "tipo_struct", "source_id"])
    # Determinar si input es point o multipatch verificando geometry type
    desc = arcpy.Describe(input_fc)
    geom_type = desc.shapeType  # "Point", "Multipatch", etc.
    with arcpy.da.SearchCursor(input_fc, [id_field, "SHAPE@"]) as s:
        for oid, geom in s:
            if geom_type == "Point":
                # si ya tiene Z, mantenlo; si no, elevar a Z = 0 (o puedes calcular DEM Z)
                p = geom
                if p.firstPoint.Z is None:
                    # intentar obtener Z desde DEM no implementado aquí; ponemos Z=0
                    p3 = arcpy.PointGeometry(arcpy.Point(p.firstPoint.X, p.firstPoint.Y, 0), sr, True)
                else:
                    p3 = p
                insert.insertRow([p3, tipo_label, oid])
            else:
                # multipatch u otro: tomar centroid XY y Z promedio de vértices
                verts = []
                for part in geom:
                    for pnt in part:
                        if pnt is None:
                            continue
                        z = pnt.Z if hasattr(pnt, "Z") and pnt.Z is not None else 0.0
                        verts.append((pnt.X, pnt.Y, z))
                if len(verts) == 0:
                    continue
                avg_x = sum(v[0] for v in verts) / len(verts)
                avg_y = sum(v[1] for v in verts) / len(verts)
                #avg_z = sum(v[2] for v in verts) / len(verts)
                max_z = max(v[2] for v in verts) + 0.5
                pgeom = arcpy.PointGeometry(arcpy.Point(avg_x, avg_y, max_z), sr, True)
                insert.insertRow([pgeom, tipo_label, oid])
    del insert


In [22]:
if puntos_lum and arcpy.Exists(puntos_lum):
    add_point_structures(puntos_lum, "luminaria")
else:
    # si lum vienen como multipatch
    if arcpy.Exists(multipatch_lum):
        if not arcpy.Exists(multipatch_lum):
            lum_sp = ensure_singlepart(multipatch_lum, "luminarias_singlepart")
            add_point_structures(lum_sp, "luminaria")
        else:
            add_point_structures("luminarias_singlepart", "luminaria")

In [23]:
if puntos_sem and arcpy.Exists(puntos_sem):
    add_point_structures(puntos_sem, "semaforo")
else:
    if arcpy.Exists(multipatch_sem):
        if not arcpy.Exists(multipatch_sem):
            sem_sp = ensure_singlepart(multipatch_sem, "semaforos_singlepart")
            add_point_structures(sem_sp, "semaforo")
        else:
            add_point_structures("semaforos_singlepart", "semaforo")